In [34]:
from markdown_it.rules_block import reference
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset

checkpoint = 'bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
raw_dataset = load_dataset("nyu-mll/glue", "mrpc")

In [35]:
def tokenize(dataset, tokenizer):
    return tokenizer(
        dataset['sentence1'],
        dataset['sentence2'],
        truncation=True
    )

In [36]:
tokenized_dataset = raw_dataset.map(tokenize, batched=True, fn_kwargs={'tokenizer': tokenizer})
tokenized_dataset = tokenized_dataset.remove_columns(['sentence1', 'sentence2', 'idx']).rename_column('label', 'labels')
tokenized_dataset.set_format('torch')

In [37]:
collator = DataCollatorWithPadding(tokenizer)

In [38]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=tokenized_dataset['train'],
    batch_size=16,
    shuffle=True,
    collate_fn=collator
)

eval_loader = DataLoader(
    dataset=tokenized_dataset['validation'],
    batch_size=16,
    collate_fn=collator
)

In [39]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 883.85it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

In [40]:
batch = next(iter(train_loader))

In [41]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.7540, grad_fn=<NllLossBackward0>) torch.Size([16, 2])


In [42]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [43]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [44]:
# loss = outputs.loss
# optimizer.zero_grad()
# loss.backward()
# optimizer.step()

In [45]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    'linear',
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)
#learning rate decay

In [46]:
from tqdm import tqdm

progress_bar = tqdm(total=num_training_steps)

model.train()
for epoch in range(num_epochs):
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        y_pred = model(**batch)
        loss = y_pred.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)

100%|██████████| 690/690 [01:55<00:00,  5.88it/s]

In [47]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()

for batch in eval_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.inference_mode():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch['labels'])

metric.compute()

{'accuracy': 0.8382352941176471, 'f1': 0.8865979381443299}

also you can replace device switching easily using accelerator that handles it for you

In [48]:
from accelerate import Accelerator

accelerator = Accelerator()
model2 = AutoModelForSequenceClassification.from_pretrained(checkpoint)
optimizer2 = AdamW(model2.parameters(), lr=5e-5)
train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_loader, eval_loader, model2, optimizer2
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 985.80it/s][A
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [49]:
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [54]:
progress_bar = tqdm(
    total=num_epochs * len(train_dl),
)

  6%|▌         | 41/690 [00:09<02:27,  4.40it/s]


In [55]:
model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

100%|██████████| 690/690 [02:13<00:00,  4.92it/s]

In [56]:
metric = evaluate.load("glue", "mrpc")
model.eval()

with torch.inference_mode():
    for batch in eval_dl:
        outputs = model(**batch)
        predictions = outputs.logits.argmax(dim=-1)

        predictions, labels = accelerator.gather_for_metrics(
            (predictions, batch["labels"])
        )

        metric.add_batch(
            predictions=predictions.cpu(),
            references=labels.cpu(),
        )

print(metric.compute())

{'accuracy': 0.8431372549019608, 'f1': 0.8929765886287625}
